In [ ]:
#my draft
import numpy as np
class MyRidge:
    def __init__(self,alpha=1.0):
        self.alpha=alpha  
        self.coef_=None   
        self.intercept_=None
  
    def fit(self,X,y):    
        X=np.asarray(X)
        y=np.asarray(y)
        X_bias=np.hstack([np.ones((X.shape[0],1)),X])
        n=X_bias.shape[1]
        I=np.eye(n)
        self.w_=np.linalg.inv(
            X_bias.T@X_bias+self.alpha*I
        )@(X_bias.T@y)
        self.intercept_=self.w_[0]
        self.coef_=self.w_[1:]
        return self

    def predict(self,X):
        X=np.asarray(X)
        return X@self.coef_+self.intercept_

In [14]:
# 测试数据
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
y = np.array([3, 7, 11, 15])  # 真实关系：y = 1*X1 + 1*X2

# 初始化并拟合模型
ridge = MyRidge(alpha=0.1)
ridge.fit(X, y)

# 输出权重
print("模型权重:", ridge.coef_)  # 接近 [1, 1]

# 预测
X_test = np.array([[9, 10]])
y_pred = ridge.predict(X_test)
print("预测结果:", y_pred)  # 接近 19

模型权重: [0.99075956 1.00456271]
预测结果: [18.97626625]


In [15]:
#ai optimized  definitive version
import numpy as np

class Ridge:
    def __init__(self, alpha=1.0):
        """
        初始化Ridge回归模型（贴近sklearn接口）
        :param alpha: 正则化强度（L2正则化系数），必须≥0，默认1.0
        """
        # 超参数
        self.alpha = alpha
        # 模型参数（拟合后生成）
        self.coef_ = None  # 特征权重（不含截距）
        self.intercept_ = None  # 截距项

    def fit(self, X, y):
        """
        拟合Ridge回归模型（核心逻辑与sklearn一致）
        :param X: 特征矩阵，形状 (n_samples, n_features)
        :param y: 目标向量，形状 (n_samples,) 或 (n_samples, 1)
        :return: self（符合sklearn链式调用风格）
        """
        # 输入类型与形状校验（贴近sklearn的输入校验逻辑）
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1, 1)  # 转为列向量统一处理
        
        if X.ndim != 2:
            raise ValueError(f"X必须是二维数组，输入形状为{X.shape}")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"X样本数({X.shape[0]})与y样本数({y.shape[0]})不匹配")
        if self.alpha < 0:
            raise ValueError(f"alpha必须≥0，输入值为{self.alpha}")

        n_samples, n_features = X.shape

        # 构造带截距的扩展特征矩阵（X_ext = [1, X]，对应截距项）
        X_ext = np.hstack([np.ones((n_samples, 1)), X])  # 第一列全为1，对应截距
        
        # 正则化矩阵：仅对特征权重正则化，截距项不正则化（sklearn核心逻辑）
        reg_matrix = self.alpha * np.eye(n_features + 1)
        reg_matrix[0, 0] = 0  # 截距项（第一行/列）不参与正则化

        # Ridge核心公式：w = (X_ext^T X_ext + αI)^-1 X_ext^T y
        X_ext_T = X_ext.T
        inv_matrix = np.linalg.inv(X_ext_T @ X_ext + reg_matrix)
        w_full = inv_matrix @ X_ext_T @ y

        # 拆分截距和特征权重
        self.intercept_ = w_full[0, 0]  # 第一行是截距
        self.coef_ = w_full[1:, 0].flatten()  # 剩余行是特征权重

        return self  # 支持链式调用（如 ridge.fit(X,y).predict(X)）

    def predict(self, X):
        """
        预测新样本（与sklearn预测逻辑完全一致）
        :param X: 新特征矩阵，形状 (n_samples, n_features)
        :return: 预测值，形状 (n_samples,)
        """
        # 校验模型是否已拟合
        if self.coef_ is None or self.intercept_ is None:
            raise ValueError("模型尚未拟合，请先调用fit()方法")
        
        # 输入校验与转换
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2:
            raise ValueError(f"X必须是二维数组，输入形状为{X.shape}")
        if X.shape[1] != len(self.coef_):
            raise ValueError(f"X特征数({X.shape[1]})与模型拟合的特征数({len(self.coef_)})不匹配")

        # 预测公式：y_pred = X @ coef_ + intercept_
        y_pred = X @ self.coef_ + self.intercept_
        return y_pred.flatten()  # 转为一维数组（符合sklearn输出风格）
from sklearn.linear_model import Ridge as SklearnRidge

# 生成测试数据
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=np.float64)
y = np.array([3.1, 7.2, 11.0, 15.1], dtype=np.float64)  # 带少量噪声

# 1. 测试自定义Ridge
my_ridge = Ridge(alpha=0.1)
my_ridge.fit(X, y)
my_pred = my_ridge.predict(X)

# 2. 测试sklearn Ridge（对比）
sk_ridge = SklearnRidge(alpha=0.1, fit_intercept=True)  # 默认拟合截距
sk_ridge.fit(X, y)
sk_pred = sk_ridge.predict(X)

# 输出对比
print("=== 自定义Ridge ===")
print("截距项:", my_ridge.intercept_)
print("特征权重:", my_ridge.coef_)
print("预测值:", my_pred)

print("\n=== sklearn Ridge ===")
print("截距项:", sk_ridge.intercept_)
print("特征权重:", sk_ridge.coef_)
print("预测值:", sk_pred)

# 验证结果一致性（误差≤1e-10，因数值计算精度差异）
print("\n预测结果是否近似一致:", np.allclose(my_pred, sk_pred, atol=1e-10))

=== 自定义Ridge ===
截距项: 0.1673316708222261
特征权重: [0.9925187 0.9925187]
预测值: [ 3.14488778  7.11496259 11.08503741 15.05511222]

=== sklearn Ridge ===
截距项: 0.1673316708229553
特征权重: [0.9925187 0.9925187]
预测值: [ 3.14488778  7.11496259 11.08503741 15.05511222]

预测结果是否近似一致: True


In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
class Myridge:
    def __init__(self,alpha=1.0):
        self.alpha=alpha
        self.coef_=None
        self.intercept_=None
    def fit(self,X,y):
        X=np.asarray(X)
        y=np.asarray(y)
        one=np.ones(shape=(X.shape[0],1))
        X=np.concatenate([one,X],axis=1)
        I=np.eye(X.shape[1])
        I[0,0]=0#正则化只约束权重,不约束偏置
        w=np.linalg.inv(X.T@X+self.alpha*I)@(X.T@y)
        self.intercept_=w[0]
        self.coef_=w[1:]
    def predict(self,X):
        return X@self.coef_+self.intercept_

X,y=make_regression(n_samples=1000,n_features=5,bias=0,random_state=23,noise=2)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=323)
r=Myridge(alpha=0.5)
r.fit(X_train,y_train)

print(r.coef_)
print(r.intercept_)
y_pred=r.predict(X_test)
r2_score(y_test,y_pred)

[19.69237988 23.22386863 92.75709061 11.00754152 95.47407484]
0.10134588768495689


0.9997665332276079

In [2]:
from sklearn.linear_model import Ridge
r2=Ridge(alpha=0.5)
r2.fit(X_train,y_train)
print(r2.coef_)
print(r2.intercept_)

,alpha,0.5
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


[19.69237988 23.22386863 92.75709061 11.00754152 95.47407484]
0.10134588768497288
